In [17]:
#!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [18]:
#!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [15]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [16]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [17]:
documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(index, openai_client)

In [5]:
assistant.rag('I just discovered the course, can I still join?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [6]:
assistant.rag('how much do you charge per hour?')

'For an individual student, private 1-to-1 GCSE tutoring is **£30 per hour**.'

In [7]:
assistant.rag('Who is Wali?')

'Wali is an online GCSE and KS3 Maths tutor based in the UK. He supports the AQA, Edexcel, and OCR exam boards, and his 60-minute lessons focus on step-by-step explanations, helping weak students regain confidence, improving exam technique, and targeting specific topic weaknesses.'

# Failed Questions
### when using just keyword search only

In [8]:
assistant.rag('How many years experience have you been teaching in Mathematics and what level?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [9]:
assistant.rag('How many years experience have you been teaching in Mathematics?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [10]:
assistant.rag('Does Wali know how to teach pythagoras theorem?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [11]:
response = assistant.rag(
    query="Tell me about your fees and costs.",
    filter_dict={'category': 'pricing'}
)
response

'Private 1-to-1 GCSE maths tutoring costs £30 per hour. There are no hidden registration fees mentioned, so it’s just the hourly rate.\n\n'

In [18]:
import json
import pandas as pd

In [19]:
with open("knowledge-base.json", "r") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

# ==========================================
# Phase 1: Create the Knowledge Base
# ==========================================
# We only want unique context chunks to ingest into Elasticsearch.
# If we ingest duplicates, our vector search will return the same text multiple times.

# Drop duplicates based on the chunk_id
kb_df = df[['chunk_id', 'category', 'subcategory', 'mapped_context']].drop_duplicates(subset=['chunk_id'])

In [22]:
kb_df.category.unique()

<StringArray>
[          'tutor_profile',        'lesson_structure',
             'exam_boards',                 'pricing',
            'availability',           'lesson_format',
       'teaching_approach', 'objections_and_concerns',
       'business_policies',     'booking_and_contact']
Length: 10, dtype: str

In [24]:
assistant.rag(
          query='How many years experience have you been teaching in Mathematics?',
              filter_dict={'category': 'tutor_profile'}
             )

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."